In [2]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('../data/processed/dataset_clean.csv')

print('Размер датасета:', df.shape)
df.head()

Размер датасета: (113999, 21)


,Unnamed: 0,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [3]:
features = [
    'danceability',
    'energy',
    'key',
    'loudness',
    'mode',
    'speechiness',
    'acousticness',
    'instrumentalness',
    'liveness',
    'valence',
    'tempo',
    'time_signature'
]

target = 'track_genre'

In [4]:
X = df[features]
y = df[target]

print('X:', X.shape)
print('y:', y.shape)

print('\nПризнаки:')
print(X.columns.tolist())

print('\nКоличество жанров:', y.nunique())

X: (113999, 12)
y: (113999,)

Признаки:
['danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']

Количество жанров: 114


In [5]:
track_genre_count = (
    df.groupby('track_id')['track_genre']
      .nunique()
)

single_genre_ids = track_genre_count[
    track_genre_count == 1
].index

df_single = df[
    df['track_id'].isin(single_genre_ids)
].copy()

print('Размер df_single:', df_single.shape)
print('Уникальных track_id:', df_single['track_id'].nunique())
print('Количество жанров:', df_single['track_genre'].nunique())

Размер df_single: (73789, 21)
Уникальных track_id: 73441
Количество жанров: 112


In [6]:
X_single = df_single[features]
y_single = df_single[target]

print('X_single:', X_single.shape)
print('y_single:', y_single.shape)

print('\nРаспределение жанров:')
print(y_single.value_counts().describe())

X_single: (73789, 12)
y_single: (73789,)

Распределение жанров:
count     112.000000
mean      658.830357
std       271.157309
min        72.000000
25%       455.250000
50%       702.000000
75%       886.000000
max      1000.000000
Name: count, dtype: float64


После перехода к одножанровым трекам датасет стал заметно несбалансированным.

In [8]:
from sklearn.model_selection import train_test_split

# Сначала отделяем test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X_single,
    y_single,
    test_size=0.2,
    random_state=42,
    stratify=y_single
)

# Затем отделяем validation от train
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.2,
    random_state=42,
    stratify=y_train_val
)

print('Train:')
print(X_train.shape, y_train.shape)

print('\nValidation:')
print(X_val.shape, y_val.shape)

print('\nTest:')
print(X_test.shape, y_test.shape)

Train:
(47224, 12) (47224,)

Validation:
(11807, 12) (11807,)

Test:
(14758, 12) (14758,)


Делаем baseline

In [10]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

dummy = DummyClassifier(
    strategy='most_frequent'
)

dummy.fit(X_train, y_train)

y_val_pred = dummy.predict(X_val)

accuracy = accuracy_score(y_val, y_val_pred)
f1_macro = f1_score(y_val, y_val_pred, average='macro')
f1_weighted = f1_score(y_val, y_val_pred, average='weighted')

print(f'Accuracy: {accuracy:.4f}')
print(f'Macro-F1: {f1_macro:.4f}')
print(f'Weighted-F1: {f1_weighted:.4f}')

Accuracy: 0.0136
Macro-F1: 0.0002
Weighted-F1: 0.0004
